This article explores a strategy that utilizes active learning to accelerate the process of annotating text data with named entities. Active learning is a machine learning field that aims to reduce the overall amount of labeled data needed by selectively choosing data samples that offer significant understanding into the problem being tackled. This approach enables you to concentrate on labeling those data points that are most valuable for the task.

[In this instance, I utilize the [ingredient dataset](https://github.com/explosion/projects/tree/v3/tutorials/ner_food_ingredients) made available by Explosion.]{.aside}

This article does not demonstrate active learning to select lower confidence model because selectiong these samples in an named entity recognition context is little more difficult. Instead, I utilize the model's named entity predictions to **pre-annotate** entities in the data, which can then be reviewed and edited by the annotator, speeding up the annotation process.



::: {.callout}
- **[Label Studio](https://labelstud.io/)** is used for annotating data. It can be leveraged to prepare raw data or improve existing training data to get more accurate ML models."
- **[spaCy](https://spacy.io/)** is used to train a custom **named entity recognition** model, enableing us to generate named entity predictions on unlabeled data and add pre-annotations.
:::

Let's get started with annotating some entities.

## Annotating Data

First [install](https://labelstud.io/guide/install.html) Label Studio and the [Label Studio SDK](https://labelstud.io/guide/sdk.html) which allows to to interact with the API using a Python interface.

To begin, set up the connection to the Label Studio API by obtaining your API key from your user profile in Label Studio. In your script, include the following code, substituting your own values for `LABEL_STUDIO_URL` and `LABEL_STUDIO_API_KEY` which can be found under your account setting in the Label Studio interface.

In [1]:
LABEL_STUDIO_URL = 'http://localhost:8080'
LABEL_STUDIO_API_KEY = '638e606905d48b784e3fa7740405007e0e1e8fb1'
PROJECT_ID = 9

Setup an connection to the API using the Label Studio SDK Client.

In [2]:
from label_studio_sdk import Client

client = Client(url=LABEL_STUDIO_URL, api_key=LABEL_STUDIO_API_KEY)

Setup named entity recognition using the Label Studio UI and annotate some texts.

Select the project your working on based on your projects id (seen the path in label studio url). Use the the `get_labeled_task()` method to retrieve all tasks that have been completed/annotated.

In [3]:
project = client.get_project(PROJECT_ID)
labeled_tasks = project.get_labeled_tasks()

print(f"Found {len(labeled_tasks)} labeled tasks.")

Found 188 labeled tasks.


## Create a Custom NER model using spaCy

After we annotated some texts let's train our first custom spaCy model. To train a custom named entity recognition model, a specific data structure/format is required by spaCy.

```python
[
    {
      "text": "We need basil to add to the mushrooms",
      "entities": [
        (8, 12, "INGRED"),
        (28, 36, "INGRED")
      ]
    }
]
```

We need to transform the labeled tasks from Label Studio to this to this spaCy format.

In [4]:
from tqdm.notebook import tqdm


def create_training_data(labeled_tasks: dict, column: str) -> list[dict]:
    """Transform label studio annotations to spaCy format"""

    training_data: list[str] = []
    
    # Iterate over the labeled tasks
    for task in tqdm(labeled_tasks, desc="Tasks"):
        task_dict = {}
        
        # Get text from task structure
        task_dict["text"] = task["data"][column]
        task_dict["entities"] = []
        
        # Iterate over annotations (results)
        for annotations in task["annotations"]:
            for annotation in annotations["result"]:
                values = annotation["value"]
                
                # Extract start, end and label from labeled entities
                start = values["start"]
                end = values["end"]
                label = values["labels"][0]
                
                task_dict["entities"].append((start, end, label))

        training_data.append(task_dict)

    return training_data


training_data = create_training_data(labeled_tasks, "text")

Tasks:   0%|          | 0/188 [00:00<?, ?it/s]

A single training sample will look like this:

In [5]:
training_data[0]

{'text': "Where do you get the mock duck? I've only recently tried it in a restaurant and loved it. Hoisin we use for sandwich condiment mixed with sriracha. You could make those pancakes with another faux-meat. Some of those grain sausages are really good and you can slice them. The brand I buy sometimes is Field Roast. Also hoisin stir fried veggies with peanut is delicious.",
 'entities': [(26, 30, 'INGRED'),
  (90, 96, 'INGRED'),
  (138, 146, 'INGRED'),
  (191, 200, 'INGRED'),
  (222, 230, 'INGRED'),
  (318, 324, 'INGRED'),
  (349, 355, 'INGRED')]}

To create annotated data for training, spaCy utilizes the `DocBin` class. In some cases, entity spans overlap, meaning that the indices of certain entities intersect. To address this, spaCy offers a utility method called `filter_spans`.

In [6]:
import spacy

from spacy.tokens import DocBin, Span
from spacy.util import filter_spans

# Initialize a blank spacy model
nlp = spacy.blank("en")

# Initialize a DocBin object
doc_bin = DocBin()

# Iterate of training example
for example in tqdm(training_data):
    
    # Get text & entities from training data
    text = example["text"]
    labels = example["entities"]
    
    # Turn a text into a Doc object.
    doc = nlp.make_doc(text)
    
    ents: list[Span] = []
    
    # Loop over entities tuple
    for start, end, label in labels:
        
        # Create span object using start & end index
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        
        if span is None:
            continue
        
        else:
            ents.append(span)

    # Remove duplicates or overlaps
    doc.ents = filter_spans(ents)
    
    # Add document to doc bin
    doc_bin.add(doc)

# Save train dataset to disk
doc_bin.to_disk("train.spacy")

  0%|          | 0/188 [00:00<?, ?it/s]

For your specific use case, you may create a configuration file manually or generate a basic configuration using spaCy's training [quickstart page](https://spacy.io/usage/training#quickstart).

To simplify the process, we will utilize a basic configuration file from the quickstart page as our starting point. However, feel free to tweak is config to your liking.

Once you have saved the initial configuration to a file named "base_config.cfg", you can employ the "init fill-config" command to complete the missing default values.

In [7]:
!python -m spacy init fill-config /Users/maxscheijen/Developer/online-learning/base_config.cfg config.cfg

✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


We now have everything required to train our model. However, as we did not create validation/development dataset seperate from our training dataset, we will utilize the training dataset for validation purposes, which is ofcourse not an ideal approach.

In [8]:
!python -m spacy train config.cfg --output ./ --paths.train ./train.spacy --paths.dev ./train.spacy

ℹ Saving to output directory: .
ℹ Using CPU

=========================== Initializing pipeline ===========================
[2023-04-11 12:13:26,841] [INFO] Set up nlp object from config
[2023-04-11 12:13:26,854] [INFO] Pipeline: ['tok2vec', 'ner']
[2023-04-11 12:13:26,858] [INFO] Created vocabulary
[2023-04-11 12:13:29,755] [INFO] Added vectors: en_core_web_lg
[2023-04-11 12:13:35,515] [INFO] Finished initializing nlp object
[2023-04-11 12:13:36,507] [INFO] Initialized pipeline components: ['tok2vec', 'ner']
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     49.67    0.28    1.03    0.16    0.00
  2     200         16.40   1727.22   69.98   72.17   67.92    0.70
  4     400         34.90    736.29   86.95   90.16  

We will now load the best-performing model and test it on a given piece of text.

In [9]:
from spacy import displacy

nlp_ner = spacy.load("model-best")

doc_example = nlp_ner("Use some more garlic on the dish. Also grab some mushrooms")
displacy.render(doc_example, style="ent", jupyter=True)

## Add pre-annotated entities to Label Studio

Once the annotated tasks have been gathered, the unlabeled tasks should be collected to allow the machine learning model to generate predictions. We'll probaly have a large number of unlabeled samples. Therefore it can be benifical to get a random sample of unlabeld tasks to perform perdictions on. 

As previously mentioned, in a more traditional setting, one could generate predictions for all unlabeled tasks and isolate the tasks where the model exhibits the most uncertainty. Subsequently, these uncertain predictions could be sent back to Label Studio for human evaluation. By adopting this approach, active learning can expedite the labeling process as only the most challenging tasks would require labeling and the easiers tasks can be labeled automatically. See the label studio [example notebook](https://github.com/heartexlabs/label-studio-sdk/blob/506fece22b463ef36c7cdc54c41eda214d6f0992/examples/active_learning/active_learning.ipynb) for this.

In [10]:
import random

# Extract all id's for unlabeled tasks
unlabeled_tasks_ids = project.get_unlabeled_tasks_ids()

# Random selection of 50 id's
batch_ids = random.sample(unlabeled_tasks_ids, 50)

# Get 50 random unlabeled tasks
unlabeled_tasks = project.get_tasks(selected_ids=batch_ids)

unlabeled_tasks[0]

{'id': 24690,
 'predictions': [],
 'annotations': [],
 'drafts': [],
 'annotators': [],
 'inner_id': 50,
 'cancelled_annotations': 0,
 'total_annotations': 0,
 'total_predictions': 0,
 'completed_at': None,
 'annotations_results': '',
 'predictions_results': '',
 'predictions_score': None,
 'file_upload': '493ee2d8-ingredients.csv',
 'storage_filename': None,
 'annotations_ids': '',
 'predictions_model_versions': '',
 'avg_lead_time': None,
 'updated_by': [],
 'data': {'text': "As a violinist I'm going to skip on that thank you very much."},
 'meta': {},
 'created_at': '2023-04-07T15:05:50.788852Z',
 'updated_at': '2023-04-07T15:05:50.788861Z',
 'is_labeled': False,
 'overlap': 1,
 'comment_count': 0,
 'unresolved_comment_count': 0,
 'last_comment_updated_at': None,
 'project': 9,
 'comment_authors': []}

### Sending Predictions to Label Studio

Upon obtaining some predictions, it is necessary to send these predictions to the labeling interface. However, Label Studio has a specific format requirement for predictions, which is generally uniform across different labeling tasks, but may differ in some cases, such as object detection, NER, or image classification.

The expected format for Label Studio NER predictions is as follows:

```python
{
    'from_name': 'label',
    'to_name': 'text',
    'type': 'labels',
    'value': {
        'start': 0
        'end': 6,
        'text': 'This is a sample text',
        'labels': ['PERSON']
    }
}
```

Specify a model version to identify the most recent set of predictions, in this example, determined by the quantity of data utilized to retrain the model. However, any unique and arbitrary name can be used for this purpose. Although setting the model version is an optional step, it can be beneficial in an Active Learning scenario, allowing for control over which model is displayed in the subsequent iteration.

I chose to use a datetime stamp and the number of labeled tasks.

In [11]:
from datetime import datetime

now = datetime.now()
dt_string = now.strftime("%Y-%m-%d %H:%M")

model_version = f'{dt_string}_{len(labeled_tasks)}'
model_version

'2023-04-11 14:38_188'

Structure the predictions/pre-annotations in the required format and incorporate them into each unlabeled task.

In [12]:
for i, unlabeled_task in enumerate(tqdm(unlabeled_tasks)):
    
    # Extract text of unlabeld task
    text = unlabeled_task["data"]["text"]
    
    # Create Document based on unlabeld task text
    doc = nlp_ner(text)
    
    # Check if entities are found
    if len(doc.ents) == 0:
        continue
    
    # If entities are found send to label studio
    else:
        results: list[dict] = []

        for ent in doc.ents:
            # Excepected label studio prediction format
            result = {
                'from_name': 'label',
                'to_name': 'text',
                'type': 'labels',
                'value': {
                    'start': ent.start_char,
                    'end': ent.end_char,
                    'text': str(ent),
                    'labels': [ent.label_]
                }
            }
            
            # Store ent predictions in list
            results.append(result)
        
        # Create a prediction for a unlabeld task
        project.create_prediction(
            task_id=unlabeled_task['id'],
            result=results,
            model_version=model_version
        )

print(f'{i} tasks have been preannotated/tagged with model predictions')

  0%|          | 0/50 [00:00<?, ?it/s]

49 tasks have been preannotated/tagged with model predictions


Finally, modify the Label Studio settings to utilize the recently generated model version for showcasing pre-annotated tasks to annotators. The annotators can use filters in the data view the label studio interface to filter on model version to get the latest predictions.

In [13]:
project.set_model_version(model_version)

By repeating this process several times, you will notice an improvement in the quality of pre-annotations, which will ultimately decrease the time required for each annotation task.

## Conclusion

And that concludes the process! With the Label Studio SDK and custom spaCy models, constructing an active learning loop for named entity recognition becomes a hassle-free and reproducible task. Additionally, Label Studio Webhooks can be utilized for event-driven active learning, which is a more advanced approach to further enhance the process.